# CAVAS — Deep Learning Model Evaluation
### IDS Intrusion Detection: TabNet (Tabular) + TFT (Time Series)

**Dual prediction targets:**
- `label_generic` → Binary classification (benign / malicious)
- `Label` → Multiclass classification (benign / attack type)

**Models:**
- **TabNet** — sequential attention, native feature importance
- **Temporal Fusion Transformer (TFT)** — variable selection networks + temporal attention

**Hyperparameter tuning:** Optuna (TPE sampler)

In [1]:
LOCAL_RUN = True
RANDOM_SEED = 42

## 0. Configuration & Setup

In [2]:
!pip install -q pytorch-tabnet pytorch-forecasting pytorch-lightning optuna optuna-integration scikit-learn pandas pyarrow


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import os, subprocess, warnings, json
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob

# Spark (used only for initial load of large parquet)
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer

# Sklearn utilities
from sklearn.model_selection  import train_test_split
from sklearn.preprocessing    import StandardScaler, LabelEncoder
from sklearn.metrics          import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score, matthews_corrcoef
)

# TabNet
from pytorch_tabnet.tab_model import TabNetClassifier

# TFT
import torch
import pytorch_lightning as pl
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data            import GroupNormalizer
from pytorch_forecasting.metrics         import CrossEntropy
from pytorch_lightning.callbacks         import EarlyStopping, ModelCheckpoint

# Optuna
import optuna
from optuna.integration import PyTorchLightningPruningCallback
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings('ignore')
pl.seed_everything(RANDOM_SEED)
print("All imports OK")

/home/cava/Documents/Repos/python/pucktrick/.venv/lib/python3.10/site-packages/pytorch_forecasting/models/base/_base_model.py:30: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
Seed set to 42


All imports OK


In [4]:
# ─────────────────────────────────────────────────────────────────────
# CONFIGURATION — change only here
# ─────────────────────────────────────────────────────────────────────
PATH_IMG = "images"

if LOCAL_RUN:
    PATH = "DATASETS"
    PERCENTAGE_TO_USE = 0.03          # fraction of data for local runs
    N_OPTUNA_TRIALS   = 20           # fewer trials locally
    TABNET_MAX_EPOCHS = 50
    TFT_MAX_EPOCHS    = 30
else:
    PATH = "file:///home/PuckTrickadmin/DATASETS"
    PERCENTAGE_TO_USE = 1.0
    N_OPTUNA_TRIALS   = 100
    TABNET_MAX_EPOCHS = 200
    TFT_MAX_EPOCHS    = 100

RANDOM_SEED  = 42
TEST_SIZE    = 0.2
VAL_SIZE     = 0.1          # fraction of TRAIN used for validation

os.makedirs(PATH_IMG, exist_ok=True)
os.makedirs("models",  exist_ok=True)

In [5]:
# ── Spark session (LOCAL / SERVER) ──────────────────────────────────
if LOCAL_RUN:
    java_home = os.environ.get('JAVA_HOME', '')
    if not java_home:
        try:
            java_path = subprocess.check_output(['which', 'java'], text=True).strip()
            os.environ['JAVA_HOME'] = os.path.dirname(os.path.dirname(os.path.realpath(java_path)))
        except subprocess.CalledProcessError:
            print("⚠️  Java not found — run: sudo apt install default-jdk")

    os.environ['PYSPARK_PYTHON']        = 'python3'
    os.environ['PYSPARK_DRIVER_PYTHON'] = 'python3'

    spark = SparkSession.builder \
        .appName("CAVAS_Models") \
        .master("local[*]") \
        .config("spark.driver.memory",          "24g") \
        .config("spark.driver.host",            "localhost") \
        .config("spark.ui.showConsoleProgress", "false") \
        .getOrCreate()
else:
    MASTER_URL  = "spark://10.0.1.8:7077"
    DRIVER_HOST = "10.0.1.8"

    spark = SparkSession.builder \
        .appName("CAVAS_Models") \
        .master(MASTER_URL) \
        .config("spark.submit.deployMode",      "client") \
        .config("spark.executor.instances",     "4") \
        .config("spark.executor.cores",         "4") \
        .config("spark.executor.memory",        "13g") \
        .config("spark.driver.memory",          "8g") \
        .config("spark.driver.host",            DRIVER_HOST) \
        .config("spark.driver.bindAddress",     DRIVER_HOST) \
        .config("spark.sql.shuffle.partitions", "32") \
        .getOrCreate()
    spark.sparkContext.setLogLevel("WARN")

print(f"✅  Spark {spark.version} ready")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 00:40:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅  Spark 3.5.0 ready


## 1. Data Loading

In [6]:
# ── Load parquet ─────────────────────────────────────────────────────
sdf_full = spark.read.parquet(f'{PATH}/all_elaborated.parquet')

if PERCENTAGE_TO_USE < 1.0:
    # Stratified sampling per label_generic (la label binaria, più semplice da stratificare)
    # Calcola la fraction per ogni strato e campiona separatamente, poi unisce
    fractions = {
        row['label_generic']: PERCENTAGE_TO_USE
        for row in sdf_full.select('label_generic').distinct().collect()
    }
    sdf = sdf_full.sampleBy('label_generic', fractions=fractions, seed=RANDOM_SEED)
    print(f"📦  Stratified {PERCENTAGE_TO_USE*100:.0f}% sample → {sdf.count():,} rows")
else:
    sdf = sdf_full
    print(f"📦  Full dataset → {sdf.count():,} rows")

print(f"🧮  Columns: {len(sdf.columns)}")
sdf.printSchema()

📦  Stratified 3% sample → 484,847 rows
🧮  Columns: 46
root
 |-- Dst Port: string (nullable = true)
 |-- Protocol: string (nullable = true)
 |-- Timestamp: timestamp (nullable = true)
 |-- Flow Duration: string (nullable = true)
 |-- Tot Fwd Pkts: string (nullable = true)
 |-- Tot Bwd Pkts: string (nullable = true)
 |-- TotLen Fwd Pkts: string (nullable = true)
 |-- Fwd Pkt Len Max: string (nullable = true)
 |-- Fwd Pkt Len Min: string (nullable = true)
 |-- Fwd Pkt Len Mean: string (nullable = true)
 |-- Bwd Pkt Len Max: string (nullable = true)
 |-- Bwd Pkt Len Min: string (nullable = true)
 |-- Bwd Pkt Len Mean: string (nullable = true)
 |-- Flow Byts/s: string (nullable = true)
 |-- Flow Pkts/s: string (nullable = true)
 |-- Flow IAT Mean: string (nullable = true)
 |-- Flow IAT Std: string (nullable = true)
 |-- Flow IAT Max: string (nullable = true)
 |-- Bwd IAT Tot: string (nullable = true)
 |-- Bwd IAT Mean: string (nullable = true)
 |-- Bwd IAT Std: string (nullable = true)
 |--

## 2. Preprocessing — Normalization & Encoding

In [7]:
# ── Feature metadata from analysis ──────────────────────────────────
# Features ordered by relevance from your CSV exports
# (union of both CSV files, deduplicated)
FEATURES_LABEL_GENERIC = [
    'Tot Fwd Pkts', 'Bwd Pkts/s', 'TotLen Fwd Pkts', 'Fwd Seg Size Min',
    'Init Fwd Win Byts', 'Fwd Pkts/s', 'Bwd IAT Mean', 'Fwd Pkt Len Mean',
    'URG Flag Cnt', 'Bwd IAT Min', 'Bwd Pkt Len Min', 'Fwd Pkt Len Max',
    'Pkt Len Min', 'Pkt Len Mean', 'Fwd Pkt Len Min', 'Bwd Pkt Len Mean',
    'Bwd Pkt Len Max', 'Init Bwd Win Byts', 'Flow IAT Std', 'Bwd IAT Max',
    'Flow Duration', 'Flow IAT Max', 'Bwd IAT Std', 'Flow IAT Mean',
    'Bwd IAT Tot', 'Idle Min', 'Down/Up Ratio', 'Fwd PSH Flags',
    'Fwd URG Flags', 'Pkt Len Var', 'Active Max', 'Active Mean',
    'Active Min', 'Active Std', 'FIN Flag Cnt', 'Tot Bwd Pkts'
]

# Feature types from your analysis
CATEGORICAL_FEATURES = ['Fwd Seg Size Min', 'Pkt Len Min', 'Protocol']
BINARY_FEATURES      = ['FIN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'Fwd URG Flag', 'Fwd PSH Flag']

# All features used for modelling
ALL_FEATURES = sdf_full.columns

# ✅ corretto
EXCLUDE_FROM_FEATURES = {'Label', 'label_generic', 'Timestamp'}

ALL_FEATURES = [c for c in sdf_full.columns if c not in EXCLUDE_FROM_FEATURES]

CONTINUOUS_FEATURES = [
    f for f in ALL_FEATURES
    if f not in CATEGORICAL_FEATURES and f not in BINARY_FEATURES
]

# Verify all expected features exist in the dataset
available = set(sdf.columns)
missing   = [f for f in ALL_FEATURES if f not in available]
if missing:
    print(f"⚠️  Missing features in dataset: {missing}")
    ALL_FEATURES = [f for f in ALL_FEATURES if f not in missing]
else:
    print(f"✅  All {len(ALL_FEATURES)} features found in dataset")

✅  All 43 features found in dataset


In [8]:
def label_encoding_spark(sdf):
    """Add label_generic_enc (0/1) and Label_enc (integer) to the Spark DataFrame.
    Returns (sdf_encoded, label_classes) where label_classes maps index → Label name.
    """
    # ── Binary: label_generic is already 0/1 → cast to int ──────────
    sdf = sdf.withColumn('label_generic_enc', col('label_generic').cast('int'))

    # ── Multiclass: Label → integer index via StringIndexer ──────────
    indexer = StringIndexer(inputCol='Label', outputCol='Label_enc', handleInvalid='keep')
    model = indexer.fit(sdf)
    sdf = model.transform(sdf)
    sdf = sdf.withColumn('Label_enc', col('Label_enc').cast('int'))

    # Store ordered label list: index 0 → labels[0], etc.
    label_classes = list(model.labels)

    n_binary = sdf.select('label_generic_enc').distinct().count()
    print(f"✅  label_generic_enc: {n_binary} classes | Label_enc: {len(label_classes)} classes")
    print(f"    Label mapping: { {i: l for i, l in enumerate(label_classes)} }")
    return sdf, label_classes

In [9]:
def preprocess_to_pandas(sdf, continuous_features, categorical_features, binary_features):
    """
    Convert Spark → Pandas and clean dtypes.
    
    - Continuous: cast to float64 (scaling done AFTER train/test split to avoid leakage)
    - Categorical: integer-encoded via LabelEncoder (TabNet/TFT handle them natively)
    - Binary: cast to int
    
    ⚠ NO one-hot encoding:
      • TabNet uses cat_idxs / cat_dims natively
      • TFT uses time_varying_known_categoricals
      • Feature importance stays traceable to the original feature name
    """
    print("⏳  Converting Spark → Pandas ...")
    pdf = sdf.toPandas()
    print(f"📊  Shape: {pdf.shape}")

    available = set(pdf.columns)
    cont_cols = [c for c in continuous_features if c in available]
    cat_cols  = [c for c in categorical_features if c in available]
    bin_cols  = [c for c in binary_features if c in available]

    # ── Continuous → float64 ──────────────────────────────────────────
    for c in cont_cols:
        pdf[c] = pd.to_numeric(pdf[c], errors='coerce')
    pdf[cont_cols] = pdf[cont_cols].fillna(0.0)

    # ── Categorical → integer codes ──────────────────────────────────
    cat_encoders = {}
    cat_dims = {}
    for c in cat_cols:
        le = LabelEncoder()
        pdf[c] = le.fit_transform(pdf[c].astype(str))
        cat_encoders[c] = le
        cat_dims[c] = len(le.classes_)

    # ── Binary → int ─────────────────────────────────────────────────
    for c in bin_cols:
        pdf[c] = pd.to_numeric(pdf[c], errors='coerce').fillna(0).astype(int)

    print(f"✅  Preprocessed: {len(cont_cols)} continuous | {len(cat_cols)} categorical | {len(bin_cols)} binary")
    return pdf, cat_encoders, cat_dims

In [10]:
# ── Preprocess & cache (Pandas-based, fast) ──────────────────────────
PREPROCESSED_PATH = f"{PATH}/preprocessed_model_{int(PERCENTAGE_TO_USE*100)}pct.parquet"

if os.path.exists(PREPROCESSED_PATH):
    pdf_encoded = pd.read_parquet(PREPROCESSED_PATH)

    # Rebuild metadata from cached file
    EXCLUDE = {'Label', 'label_generic', 'Label_enc', 'label_generic_enc', 'Timestamp'}
    FINAL_FEATURES = [c for c in pdf_encoded.columns if c not in EXCLUDE]

    # Recover label_classes mapping from data
    _lbl_map = (pdf_encoded[['Label', 'Label_enc']]
                .drop_duplicates()
                .sort_values('Label_enc')
                .set_index('Label_enc')['Label']
                .to_dict())
    label_classes = [_lbl_map[i] for i in range(len(_lbl_map))]

    # Categorical metadata for TabNet
    CAT_FEATURE_NAMES = [c for c in CATEGORICAL_FEATURES if c in FINAL_FEATURES]
    CAT_IDXS = [FINAL_FEATURES.index(c) for c in CAT_FEATURE_NAMES]
    CAT_DIMS = [int(pdf_encoded[c].nunique()) for c in CAT_FEATURE_NAMES]

    print(f"✅  Cache loaded: {len(pdf_encoded):,} rows, {len(FINAL_FEATURES)} features")
else:
    print("⚠️  No cache found — running preprocessing ...")
    sdf_enc, label_classes = label_encoding_spark(sdf)
    pdf_encoded, cat_encoders, cat_dims_dict = preprocess_to_pandas(
        sdf_enc, CONTINUOUS_FEATURES, CATEGORICAL_FEATURES, BINARY_FEATURES
    )

    EXCLUDE = {'Label', 'label_generic', 'Label_enc', 'label_generic_enc', 'Timestamp'}
    FINAL_FEATURES = [c for c in pdf_encoded.columns if c not in EXCLUDE]
    CAT_FEATURE_NAMES = [c for c in CATEGORICAL_FEATURES if c in FINAL_FEATURES]
    CAT_IDXS = [FINAL_FEATURES.index(c) for c in CAT_FEATURE_NAMES]
    CAT_DIMS = [cat_dims_dict[c] for c in CAT_FEATURE_NAMES]

    pdf_encoded.to_parquet(PREPROCESSED_PATH, index=False)
    print(f"💾  Saved to {PREPROCESSED_PATH}")

# ── Global counters ──────────────────────────────────────────────────
n_classes_multi = len(label_classes)

print(f"\n🧮  Features: {len(FINAL_FEATURES)} | Binary classes: 2 | Multi classes: {n_classes_multi}")
print(f"📌  Label mapping: { {i: l for i, l in enumerate(label_classes)} }")
if CAT_IDXS:
    print(f"📌  Categorical for TabNet: {list(zip(CAT_FEATURE_NAMES, CAT_IDXS, CAT_DIMS))}")
else:
    print("📌  No categorical features detected in final feature set")

✅  Cache loaded: 484,847 rows, 43 features

🧮  Features: 43 | Binary classes: 2 | Multi classes: 15
📌  Label mapping: {0: 'Benign', 1: 'DDOS attack-HOIC', 2: 'DDoS attacks-LOIC-HTTP', 3: 'DoS attacks-Hulk', 4: 'Bot', 5: 'FTP-BruteForce', 6: 'SSH-Bruteforce', 7: 'Infilteration', 8: 'DoS attacks-SlowHTTPTest', 9: 'DoS attacks-GoldenEye', 10: 'DoS attacks-Slowloris', 11: 'DDOS attack-LOIC-UDP', 12: 'Brute Force -Web', 13: 'Brute Force -XSS', 14: 'SQL Injection'}
📌  Categorical for TabNet: [('Fwd Seg Size Min', 37, 9), ('Pkt Len Min', 26, 188), ('Protocol', 1, 3)]


In [11]:
# Diagnostica: cosa c'è davvero nel dataframe?
print("Tutte le colonne nel pdf_encoded:")
for i, c in enumerate(pdf_encoded.columns):
    is_in_exclude = c in EXCLUDE
    is_in_final = c in FINAL_FEATURES
    print(f"  {i:2d}. '{c}' → EXCLUDE={is_in_exclude}, FINAL={is_in_final}")

print(f"\nTotal columns: {len(pdf_encoded.columns)}")
print(f"FINAL_FEATURES: {len(FINAL_FEATURES)}")
print(f"EXCLUDE matches: {[c for c in pdf_encoded.columns if c in EXCLUDE]}")
print(f"NOT excluded but look like targets: {[c for c in FINAL_FEATURES if 'label' in c.lower() or 'Label' in c]}")

Tutte le colonne nel pdf_encoded:
   0. 'Dst Port' → EXCLUDE=False, FINAL=True
   1. 'Protocol' → EXCLUDE=False, FINAL=True
   2. 'Timestamp' → EXCLUDE=True, FINAL=False
   3. 'Flow Duration' → EXCLUDE=False, FINAL=True
   4. 'Tot Fwd Pkts' → EXCLUDE=False, FINAL=True
   5. 'Tot Bwd Pkts' → EXCLUDE=False, FINAL=True
   6. 'TotLen Fwd Pkts' → EXCLUDE=False, FINAL=True
   7. 'Fwd Pkt Len Max' → EXCLUDE=False, FINAL=True
   8. 'Fwd Pkt Len Min' → EXCLUDE=False, FINAL=True
   9. 'Fwd Pkt Len Mean' → EXCLUDE=False, FINAL=True
  10. 'Bwd Pkt Len Max' → EXCLUDE=False, FINAL=True
  11. 'Bwd Pkt Len Min' → EXCLUDE=False, FINAL=True
  12. 'Bwd Pkt Len Mean' → EXCLUDE=False, FINAL=True
  13. 'Flow Byts/s' → EXCLUDE=False, FINAL=True
  14. 'Flow Pkts/s' → EXCLUDE=False, FINAL=True
  15. 'Flow IAT Mean' → EXCLUDE=False, FINAL=True
  16. 'Flow IAT Std' → EXCLUDE=False, FINAL=True
  17. 'Flow IAT Max' → EXCLUDE=False, FINAL=True
  18. 'Bwd IAT Tot' → EXCLUDE=False, FINAL=True
  19. 'Bwd IAT Mean' → E

### 2.1 Prepare arrays for modelling

In [12]:
# ── Data is already in Pandas from preprocessing ─────────────────────
X = pdf_encoded[FINAL_FEATURES].values.astype(np.float64)  # float64 first to detect issues
y_binary = pdf_encoded['label_generic_enc'].values.astype(int)
y_multi  = pdf_encoded['Label_enc'].values.astype(int)

# ── Clean inf / extreme values ───────────────────────────────────────
n_inf = np.isinf(X).sum()
n_nan = np.isnan(X).sum()
if n_inf > 0 or n_nan > 0:
    print(f"⚠️  Found {n_inf} inf and {n_nan} NaN values → replacing with 0")
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

# Clip extreme outliers that overflow float32 range
f32_max = np.finfo(np.float32).max
n_clipped = (np.abs(X) > f32_max).sum()
if n_clipped > 0:
    print(f"⚠️  Clipping {n_clipped} values exceeding float32 range")
    X = np.clip(X, -f32_max, f32_max)

X = X.astype(np.float32)

print(f"📊  X shape: {X.shape}")
print(f"    y_binary: {np.unique(y_binary, return_counts=True)}")
print(f"    y_multi:  {len(np.unique(y_multi))} classes (n_classes_multi={n_classes_multi})")

⚠️  Found 2114 inf and 0 NaN values → replacing with 0
📊  X shape: (484847, 43)
    y_binary: (array([0, 1]), array([402329,  82518]))
    y_multi:  15 classes (n_classes_multi=15)


In [13]:
# ── Train / Val / Test split + StandardScaler ─────────────────────────
MIN_SAMPLES_PER_CLASS = 5  # minimum samples per multiclass label to allow stratified split

def make_splits(X, y, test_size=TEST_SIZE, val_size=VAL_SIZE, seed=RANDOM_SEED):
    """Returns (X_train, X_val, X_test, y_train, y_val, y_test)"""
    X_tr, X_test, y_tr, y_test = train_test_split(
        X, y, test_size=test_size, random_state=seed, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_tr, y_tr,
        test_size=val_size / (1 - test_size),
        random_state=seed, stratify=y_tr
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

# ── Splits — BINARY (no issues, only 2 classes) ──────────────────────
X_tr_bin, X_val_bin, X_te_bin, y_tr_bin, y_val_bin, y_te_bin = make_splits(X, y_binary)

# ── Splits — MULTICLASS (filter out rare classes first) ──────────────
classes, counts = np.unique(y_multi, return_counts=True)
rare_classes = classes[counts < MIN_SAMPLES_PER_CLASS]

if len(rare_classes) > 0:
    rare_labels = [label_classes[c] for c in rare_classes]
    rare_counts = counts[counts < MIN_SAMPLES_PER_CLASS]
    print(f"⚠️  Dropping {len(rare_classes)} rare classes (< {MIN_SAMPLES_PER_CLASS} samples):")
    for lbl, cls, cnt in zip(rare_labels, rare_classes, rare_counts):
        print(f"    class {cls} ({lbl}): {cnt} sample(s)")

    keep_mask = ~np.isin(y_multi, rare_classes)
    X_multi_clean = X[keep_mask]
    y_multi_clean = y_multi[keep_mask]
    print(f"    Kept {len(y_multi_clean):,} / {len(y_multi):,} rows ({len(y_multi_clean)/len(y_multi)*100:.1f}%)")
else:
    X_multi_clean = X
    y_multi_clean = y_multi
    print("✅  All multiclass labels have enough samples for stratified split")

X_tr_mul, X_val_mul, X_te_mul, y_tr_mul, y_val_mul, y_te_mul = make_splits(
    X_multi_clean, y_multi_clean
)

# ── StandardScaler on continuous features only (fit on TRAIN → no leakage) ──
cont_idxs = [FINAL_FEATURES.index(c) for c in CONTINUOUS_FEATURES if c in FINAL_FEATURES]
scaler = StandardScaler()
scaler.fit(X_tr_bin[:, cont_idxs])  # fit on train only

for arr in [X_tr_bin, X_val_bin, X_te_bin, X_tr_mul, X_val_mul, X_te_mul]:
    arr[:, cont_idxs] = scaler.transform(arr[:, cont_idxs])

print(f"\nBinary  → Train: {len(X_tr_bin):,} | Val: {len(X_val_bin):,} | Test: {len(X_te_bin):,}")
print(f"Multi   → Train: {len(X_tr_mul):,} | Val: {len(X_val_mul):,} | Test: {len(X_te_mul):,}")
print(f"Scaled {len(cont_idxs)} continuous features (fit on train only, no leakage)")

⚠️  Dropping 1 rare classes (< 5 samples):
    class 14 (SQL Injection): 1 sample(s)
    Kept 484,846 / 484,847 rows (100.0%)

Binary  → Train: 339,392 | Val: 48,485 | Test: 96,970
Multi   → Train: 339,391 | Val: 48,485 | Test: 96,970
Scaled 35 continuous features (fit on train only, no leakage)


## 3. TabNet — Tabular Model

### 3.1 Hyperparameter Tuning (Optuna)

**Key TabNet hyperparameters:**
| Param | Meaning |
|---|---|
| `N_a` / `N_d` | Width of attention + decision steps (usually equal) |
| `N_steps` | Number of sequential attention steps |
| `gamma` | Sparsity regularisation coefficient |
| `lambda_sparse` | Feature sparsity penalty |
| `lr` | Learning rate |

In [14]:
def tabnet_objective(trial, X_train, X_val, y_train, y_val, n_classes):
    """Optuna objective for TabNet — returns validation MCC."""

    # ── Sample hyperparameters ────────────────────────────────────────
    N_a      = trial.suggest_categorical('N_a',      [8, 16, 32, 64, 128])
    N_steps  = trial.suggest_int('N_steps',          3, 10)
    gamma    = trial.suggest_float('gamma',          1.0, 2.0, step=0.1)
    lambda_s = trial.suggest_float('lambda_sparse',  1e-4, 1e-1, log=True)
    lr       = trial.suggest_float('lr',             1e-4, 1e-2, log=True)
    batch_sz = trial.suggest_categorical('batch_size', [256, 512, 1024, 2048])
    mask_type= trial.suggest_categorical('mask_type', ['sparsemax', 'entmax'])

    clf = TabNetClassifier(
        n_d=N_a, n_a=N_a,
        n_steps=N_steps,
        gamma=gamma,
        lambda_sparse=lambda_s,
        cat_idxs=CAT_IDXS if CAT_IDXS else [],
        cat_dims=CAT_DIMS if CAT_DIMS else [],
        cat_emb_dim=1,
        optimizer_params=dict(lr=lr),
        mask_type=mask_type,
        verbose=0,
        seed=RANDOM_SEED,
    )

    clf.fit(
        X_train, y_train,
        eval_set    =[(X_val, y_val)],
        eval_metric =['accuracy'],
        max_epochs  =TABNET_MAX_EPOCHS,
        patience    =10,
        batch_size  =batch_sz,
        virtual_batch_size=max(batch_sz // 4, 64),
        drop_last   =False,
    )

    preds = clf.predict(X_val)
    mcc   = matthews_corrcoef(y_val, preds)
    return mcc

In [15]:
# ── Tune TabNet — BINARY ─────────────────────────────────────────────
print("🔍  Tuning TabNet (binary: label_generic) ...")

study_tabnet_bin = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5),
    study_name='tabnet_binary'
)

study_tabnet_bin.optimize(
    lambda t: tabnet_objective(t, X_tr_bin, X_val_bin, y_tr_bin, y_val_bin, n_classes=2),
    n_trials=N_OPTUNA_TRIALS,
    timeout=36000,
    show_progress_bar=True
)

best_bin = study_tabnet_bin.best_params
print(f"\n✅  Best MCC (binary): {study_tabnet_bin.best_value:.4f}")
print(f"Best params: {best_bin}")

🔍  Tuning TabNet (binary: label_generic) ...


  0%|          | 0/20 [00:00<?, ?it/s]


Early stopping occurred at epoch 41 with best_epoch = 31 and best_val_0_accuracy = 0.99004


Best trial: 0. Best value: 0.964482:   5%|▌         | 1/20 [15:34<4:55:48, 934.14s/it, 934.14/36000 seconds]


Early stopping occurred at epoch 49 with best_epoch = 39 and best_val_0_accuracy = 0.98985


...

Best trial: 3. Best value: 0.964554:  35%|███▌      | 7/20 [5:43:05<11:11:56, 3101.26s/it, 20585.43/36000 seconds]


Early stopping occurred at epoch 37 with best_epoch = 27 and best_val_0_accuracy = 0.9901


Best trial: 7. Best value: 0.964702:  40%|████      | 8/20 [6:55:23<10:23:05, 3115.43s/it, 22456.07/36000 seconds]


[W 2026-03-03 07:35:50,452] Trial 8 failed with parameters: {'N_a': 16, 'N_steps': 6, 'gamma': 1.8, 'lambda_sparse': 0.03821129441691227, 'lr': 0.00010325337616482048, 'batch_size': 256, 'mask_type': 'entmax'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/cava/Documents/Repos/python/pucktrick/.venv/lib/python3.10/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_61716/2065917370.py", line 12, in <lambda>
    lambda t: tabnet_objective(t, X_tr_bin, X_val_bin, y_tr_bin, y_val_bin, n_classes=2),
  File "/tmp/ipykernel_61716/2481625082.py", line 27, in tabnet_objective
    clf.fit(
  File "/home/cava/Documents/Repos/python/pucktrick/.venv/lib/python3.10/site-packages/pytorch_tabnet/abstract_model.py", line 258, in fit
    self._train_epoch(train_dataloader)
  File "/home/cava/Documents/Repos/python/pucktrick/.venv/lib/python3.10/site-packages/pytorch_tabnet/abstr

KeyboardInterrupt: 

In [ ]:
# ── Tune TabNet — MULTICLASS ─────────────────────────────────────────
print("🔍  Tuning TabNet (multiclass: Label) ...")

study_tabnet_mul = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5),
    study_name='tabnet_multiclass'
)

study_tabnet_mul.optimize(
    lambda t: tabnet_objective(t, X_tr_mul, X_val_mul, y_tr_mul, y_val_mul,
                               n_classes=n_classes_multi),
    n_trials=N_OPTUNA_TRIALS,
    timeout=3600,
    show_progress_bar=True
)

best_mul = study_tabnet_mul.best_params
print(f"\n✅  Best MCC (multiclass): {study_tabnet_mul.best_value:.4f}")
print(f"Best params: {best_mul}")

: 

In [ ]:
# ─── Helper: build & train a TabNet from best params ─────────────────
def build_tabnet(params, n_classes):
    return TabNetClassifier(
        n_d=params['N_a'],  n_a=params['N_a'],
        n_steps=params['N_steps'],
        gamma=params['gamma'],
        lambda_sparse=params['lambda_sparse'],
        cat_idxs=CAT_IDXS if CAT_IDXS else [],
        cat_dims=CAT_DIMS if CAT_DIMS else [],
        cat_emb_dim=1,
        optimizer_params=dict(lr=params['lr']),
        mask_type=params['mask_type'],
        verbose=1,
        seed=RANDOM_SEED,
    )

def train_final_tabnet(params, X_train, X_val, X_test, y_train, y_val, y_test,
                       label_name='binary'):
    clf = build_tabnet(params, n_classes=len(np.unique(y_train)))
    clf.fit(
        X_train, y_train,
        eval_set   =[(X_val, y_val)],
        eval_metric=['accuracy'],
        max_epochs =TABNET_MAX_EPOCHS * 2,  # more epochs for final model
        patience   =20,
        batch_size =params['batch_size'],
        virtual_batch_size=max(params['batch_size']//4, 64),
        drop_last  =False,
    )
    clf.save_model(f'models/tabnet_{label_name}')
    return clf

: 

In [ ]:
# ── Train final TabNet models ─────────────────────────────────────────
print("🏋️  Training final TabNet — BINARY ...")
tabnet_bin = train_final_tabnet(
    best_bin, X_tr_bin, X_val_bin, X_te_bin,
                y_tr_bin, y_val_bin, y_te_bin,
    label_name='binary'
)

print("\n🏋️  Training final TabNet — MULTICLASS ...")
tabnet_mul = train_final_tabnet(
    best_mul, X_tr_mul, X_val_mul, X_te_mul,
                y_tr_mul, y_val_mul, y_te_mul,
    label_name='multiclass'
)

: 

### 3.2 TabNet — Evaluation & Feature Importance

In [ ]:
def evaluate_model(model, X_test, y_test, label_name, class_names=None):
    """Print metrics and return results dict."""
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)

    is_binary = (len(np.unique(y_test)) == 2)

    acc = accuracy_score(y_test, preds)
    mcc = matthews_corrcoef(y_test, preds)
    f1  = f1_score(y_test, preds, average='binary' if is_binary else 'macro')
    try:
        auc = roc_auc_score(
            y_test, proba if not is_binary else proba[:, 1],
            multi_class='ovr' if not is_binary else 'raise'
        )
    except Exception:
        auc = float('nan')

    print(f"\n{'='*50}")
    print(f"  {label_name.upper()} — Test Metrics")
    print(f"{'='*50}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  F1       : {f1:.4f}")
    print(f"  MCC      : {mcc:.4f}")
    print(f"  AUC-ROC  : {auc:.4f}")
    print(f"{'='*50}")
    print(classification_report(y_test, preds, target_names=class_names))

    return dict(accuracy=acc, f1=f1, mcc=mcc, auc=auc)

results_tabnet_bin = evaluate_model(tabnet_bin, X_te_bin, y_te_bin,
                                    'TabNet — Binary (label_generic)')
results_tabnet_mul = evaluate_model(tabnet_mul, X_te_mul, y_te_mul,
                                    'TabNet — Multiclass (Label)',
                                    class_names=label_classes)

: 

In [ ]:
# ── TabNet Global Feature Importance ─────────────────────────────────
# feature_importances_ is the mean attention mask over all training steps

def plot_tabnet_importance(model, feature_names, title, top_n=20, filename=None):
    importances = model.feature_importances_       # shape: (n_features,)
    fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
    fi_df = fi_df.sort_values('importance', ascending=False).head(top_n)

    fig, ax = plt.subplots(figsize=(10, top_n * 0.4))
    sns.barplot(data=fi_df, x='importance', y='feature', palette='viridis', ax=ax)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Attention-based importance')
    plt.tight_layout()
    if filename:
        plt.savefig(f'{PATH_IMG}/{filename}', dpi=150)
    plt.show()
    return fi_df

fi_bin = plot_tabnet_importance(
    tabnet_bin, FINAL_FEATURES,
    'TabNet Feature Importance — Binary (label_generic)',
    filename='tabnet_fi_binary.png'
)

fi_mul = plot_tabnet_importance(
    tabnet_mul, FINAL_FEATURES,
    'TabNet Feature Importance — Multiclass (Label)',
    filename='tabnet_fi_multiclass.png'
)

: 

In [ ]:
# ── Optuna visualisation — hyperparameter importance ─────────────────
fig = optuna.visualization.matplotlib.plot_param_importances(study_tabnet_bin)
plt.title('TabNet HPO importance — binary')
plt.savefig(f'{PATH_IMG}/optuna_tabnet_binary.png', dpi=150, bbox_inches='tight')
plt.show()

fig = optuna.visualization.matplotlib.plot_param_importances(study_tabnet_mul)
plt.title('TabNet HPO importance — multiclass')
plt.savefig(f'{PATH_IMG}/optuna_tabnet_multiclass.png', dpi=150, bbox_inches='tight')
plt.show()

: 

## 4. TFT — Temporal Fusion Transformer (Time Series)

### Strategy: Flow-level Time Series

Each unique **source IP** (or `Src IP + Dst IP` pair) is treated as an independent **time series**.
Flows are sorted by `Timestamp` and indexed sequentially within each series.
This captures temporal patterns in attack campaigns (port scans, DDoS bursts, etc.).

If `Timestamp` is not present, a **sequential index** is used as a proxy.

In [ ]:
# ── Build time-indexed dataframe for TFT ─────────────────────────────
pdf_tft = pdf_encoded.copy()

# ── 1. Create group ID ("series" per source IP or a hashed pair) ─────
if 'Src IP' in pdf_tft.columns and 'Dst IP' in pdf_tft.columns:
    pdf_tft['series_id'] = (
        pdf_tft['Src IP'].astype(str) + '_' + pdf_tft['Dst IP'].astype(str)
    ).astype('category').cat.codes
    print("✅  series_id from Src IP + Dst IP")
elif 'Src IP' in pdf_tft.columns:
    pdf_tft['series_id'] = pdf_tft['Src IP'].astype('category').cat.codes
    print("✅  series_id from Src IP")
else:
    # Fallback: create artificial groups of ~50 rows
    GROUP_SIZE = 50
    pdf_tft['series_id'] = np.arange(len(pdf_tft)) // GROUP_SIZE
    print(f"⚠️  No IP columns found — using artificial groups of {GROUP_SIZE} rows")

# ── 2. Create time index within each series ──────────────────────────
if 'Timestamp' in pdf_tft.columns:
    pdf_tft['Timestamp'] = pd.to_datetime(pdf_tft['Timestamp'], errors='coerce')
    pdf_tft = pdf_tft.sort_values(['series_id', 'Timestamp'])
    pdf_tft['time_idx'] = (
        pdf_tft.groupby('series_id').cumcount()
    ).astype(int)
    print("✅  time_idx from Timestamp")
else:
    pdf_tft = pdf_tft.sort_values('series_id').reset_index(drop=True)
    pdf_tft['time_idx'] = pdf_tft.groupby('series_id').cumcount().astype(int)
    print("⚠️  No Timestamp column — using sequential time_idx")

# ── 3. Keep only series with at least MIN_SEQ_LEN entries ────────────
MIN_SEQ_LEN   = 10
MAX_PRED_LEN  = 1         # predict 1 step ahead (the label of the next flow)
MAX_ENCODER_LEN = 30      # look-back window

series_counts = pdf_tft.groupby('series_id').size()
valid_series  = series_counts[series_counts >= MIN_SEQ_LEN].index
pdf_tft = pdf_tft[pdf_tft['series_id'].isin(valid_series)].reset_index(drop=True)
print(f"Valid series: {len(valid_series):,} | Total rows: {len(pdf_tft):,}")

# Cast series_id to string (required by TFT)
pdf_tft['series_id'] = pdf_tft['series_id'].astype(str)

: 

In [ ]:
# ── TFT TimeSeriesDataSet builder ─────────────────────────────────────
def make_tft_datasets(pdf, target_col, max_encoder_len=MAX_ENCODER_LEN,
                      max_pred_len=MAX_PRED_LEN, val_frac=0.2):
    """
    Build train/val TimeSeriesDataSet for TFT.
    Splits by time_idx: last val_frac of time steps go to validation.
    """
    max_time = pdf['time_idx'].max()
    cutoff   = int(max_time * (1 - val_frac))

    # TFT needs the target to be a string for classification via CrossEntropy
    pdf = pdf.copy()
    pdf[target_col] = pdf[target_col].astype(float)  # CrossEntropy needs float

    n_classes = int(pdf[target_col].nunique())

    ts_train = TimeSeriesDataSet(
        pdf[lambda d: d['time_idx'] <= cutoff],
        time_idx          = 'time_idx',
        target            = target_col,
        group_ids         = ['series_id'],
        max_encoder_length= max_encoder_len,
        max_prediction_length=max_pred_len,
        time_varying_known_reals  = [],
        time_varying_unknown_reals= [f for f in FINAL_FEATURES if f in pdf.columns],
        target_normalizer = None,
        add_relative_time_idx=True,
        add_target_scales    =False,
        add_encoder_length   =True,
    )

    ts_val = TimeSeriesDataSet.from_dataset(
        ts_train,
        pdf[lambda d: d['time_idx'] > cutoff - max_encoder_len],
        predict=True
    )

    return ts_train, ts_val, n_classes


print("Building TFT datasets — binary ...")
ts_train_bin, ts_val_bin, _ = make_tft_datasets(pdf_tft, 'label_generic_enc')

print("Building TFT datasets — multiclass ...")
ts_train_mul, ts_val_mul, n_tft_classes = make_tft_datasets(pdf_tft, 'Label_enc')

print("✅  TFT datasets ready")

: 

### 4.1 TFT Hyperparameter Tuning (Optuna)

In [ ]:
def tft_objective(trial, ts_train, ts_val, n_classes, max_epochs):
    """Optuna objective for TFT — returns validation loss (lower is better)."""

    hidden_size        = trial.suggest_categorical('hidden_size',   [16, 32, 64, 128])
    attention_head_size= trial.suggest_int('attention_head_size',   1, 4)
    dropout            = trial.suggest_float('dropout',             0.1, 0.4, step=0.05)
    hidden_continuous  = trial.suggest_categorical('hidden_continuous', [8, 16, 32])
    lr                 = trial.suggest_float('lr',                  1e-4, 1e-2, log=True)
    batch_size         = trial.suggest_categorical('batch_size',    [64, 128, 256])

    train_dl = ts_train.to_dataloader(train=True,  batch_size=batch_size, num_workers=0)
    val_dl   = ts_val.to_dataloader(  train=False, batch_size=batch_size, num_workers=0)

    pruning_cb = PyTorchLightningPruningCallback(trial, monitor='val_loss')
    early_stop = EarlyStopping(monitor='val_loss', patience=5, mode='min')

    tft = TemporalFusionTransformer.from_dataset(
        ts_train,
        learning_rate          = lr,
        hidden_size            = hidden_size,
        attention_head_size    = attention_head_size,
        dropout                = dropout,
        hidden_continuous_size = hidden_continuous,
        output_size            = n_classes,
        loss                   = CrossEntropy(),
        log_interval           = -1,
        reduce_on_plateau_patience=3,
    )

    trainer = pl.Trainer(
        max_epochs         = max_epochs,
        accelerator        = 'auto',
        enable_progress_bar= False,
        enable_model_summary=False,
        callbacks          = [pruning_cb, early_stop],
        logger             = False,
    )

    trainer.fit(tft, train_dataloaders=train_dl, val_dataloaders=val_dl)

    return trainer.callback_metrics.get('val_loss', torch.tensor(float('inf'))).item()

: 

In [ ]:
# ── Tune TFT — BINARY ────────────────────────────────────────────────
print("🔍  Tuning TFT (binary: label_generic) ...")

study_tft_bin = optuna.create_study(
    direction  = 'minimize',
    sampler    = optuna.samplers.TPESampler(seed=RANDOM_SEED),
    pruner     = optuna.pruners.HyperbandPruner(),
    study_name = 'tft_binary'
)

study_tft_bin.optimize(
    lambda t: tft_objective(t, ts_train_bin, ts_val_bin,
                             n_classes=2, max_epochs=TFT_MAX_EPOCHS),
    n_trials         = N_OPTUNA_TRIALS,
    timeout          = 7200,
    show_progress_bar= True
)

best_tft_bin = study_tft_bin.best_params
print(f"\n✅  Best val_loss (binary): {study_tft_bin.best_value:.4f}")
print(f"Best params: {best_tft_bin}")

: 

In [ ]:
# ── Tune TFT — MULTICLASS ────────────────────────────────────────────
print("🔍  Tuning TFT (multiclass: Label) ...")

study_tft_mul = optuna.create_study(
    direction  = 'minimize',
    sampler    = optuna.samplers.TPESampler(seed=RANDOM_SEED),
    pruner     = optuna.pruners.HyperbandPruner(),
    study_name = 'tft_multiclass'
)

study_tft_mul.optimize(
    lambda t: tft_objective(t, ts_train_mul, ts_val_mul,
                             n_classes=n_tft_classes, max_epochs=TFT_MAX_EPOCHS),
    n_trials         = N_OPTUNA_TRIALS,
    timeout          = 7200,
    show_progress_bar= True
)

best_tft_mul = study_tft_mul.best_params
print(f"\n✅  Best val_loss (multiclass): {study_tft_mul.best_value:.4f}")
print(f"Best params: {best_tft_mul}")

: 

In [ ]:
# ── Train final TFT models ────────────────────────────────────────────
def train_final_tft(params, ts_train, ts_val, n_classes, max_epochs, label_name):
    train_dl = ts_train.to_dataloader(train=True,  batch_size=params['batch_size'], num_workers=0)
    val_dl   = ts_val.to_dataloader(  train=False, batch_size=params['batch_size'], num_workers=0)

    checkpoint_cb = ModelCheckpoint(
        monitor='val_loss', mode='min',
        filename=f'models/tft_{label_name}_best'
    )
    early_stop = EarlyStopping(monitor='val_loss', patience=15, mode='min')

    tft = TemporalFusionTransformer.from_dataset(
        ts_train,
        learning_rate          = params['lr'],
        hidden_size            = params['hidden_size'],
        attention_head_size    = params['attention_head_size'],
        dropout                = params['dropout'],
        hidden_continuous_size = params['hidden_continuous'],
        output_size            = n_classes,
        loss                   = CrossEntropy(),
        log_interval           = 10,
        reduce_on_plateau_patience=5,
    )

    trainer = pl.Trainer(
        max_epochs  = max_epochs * 2,
        accelerator = 'auto',
        callbacks   = [checkpoint_cb, early_stop],
        logger      = True,
    )

    trainer.fit(tft, train_dataloaders=train_dl, val_dataloaders=val_dl)
    return tft, trainer


print("🏋️  Training final TFT — BINARY ...")
tft_bin, trainer_bin = train_final_tft(
    best_tft_bin, ts_train_bin, ts_val_bin,
    n_classes=2, max_epochs=TFT_MAX_EPOCHS, label_name='binary'
)

print("\n🏋️  Training final TFT — MULTICLASS ...")
tft_mul, trainer_mul = train_final_tft(
    best_tft_mul, ts_train_mul, ts_val_mul,
    n_classes=n_tft_classes, max_epochs=TFT_MAX_EPOCHS, label_name='multiclass'
)

: 

### 4.2 TFT — Feature Importance (Variable Selection Networks)

In [ ]:
def plot_tft_importance(tft_model, ts_val, label_name, top_n=20):
    """
    Extract TFT Variable Selection Network weights.
    These represent how much each variable is attended to,
    averaged across the validation set.
    """
    val_dl = ts_val.to_dataloader(train=False, batch_size=256, num_workers=0)

    # get_attention: returns dict with variable selection weights
    interpretation = tft_model.interpret_output(
        tft_model.predict(val_dl, mode='raw', return_x=True)[0],
        reduction='sum'
    )

    # encoder variable importances
    encoder_vars = tft_model.encoder_variables
    enc_weights  = interpretation['encoder_variables'].numpy()

    fi_df = pd.DataFrame({'feature': encoder_vars, 'importance': enc_weights})
    fi_df = fi_df.sort_values('importance', ascending=False).head(top_n)

    fig, ax = plt.subplots(figsize=(10, top_n * 0.45))
    sns.barplot(data=fi_df, x='importance', y='feature', palette='magma', ax=ax)
    ax.set_title(f'TFT Variable Selection Weights — {label_name}', fontsize=14)
    ax.set_xlabel('Selection weight (sum over val set)')
    plt.tight_layout()
    plt.savefig(f'{PATH_IMG}/tft_fi_{label_name}.png', dpi=150)
    plt.show()
    return fi_df

fi_tft_bin = plot_tft_importance(tft_bin, ts_val_bin, label_name='binary')
fi_tft_mul = plot_tft_importance(tft_mul, ts_val_mul, label_name='multiclass')

: 

## 5. Results Summary & Comparison

Aggregate all metrics into a single comparison table for the thesis.

In [ ]:
# ── Build comparison table ────────────────────────────────────────────
summary_rows = []

for model_name, model, X_test, y_test, task in [
    ('TabNet', tabnet_bin, X_te_bin, y_te_bin, 'Binary'),
    ('TabNet', tabnet_mul, X_te_mul, y_te_mul, 'Multiclass'),
]:
    preds = model.predict(X_test)
    summary_rows.append({
        'Model': model_name,
        'Task' : task,
        'Accuracy': accuracy_score(y_test, preds),
        'F1 (macro)': f1_score(y_test, preds, average='binary' if task=='Binary' else 'macro'),
        'MCC': matthews_corrcoef(y_test, preds),
    })

# Note: TFT evaluation requires val dataloader; reported via val_loss from trainer
for trainer, task in [(trainer_bin, 'Binary'), (trainer_mul, 'Multiclass')]:
    val_loss = trainer.callback_metrics.get('val_loss', torch.tensor(float('nan'))).item()
    summary_rows.append({
        'Model': 'TFT',
        'Task' : task,
        'Accuracy': float('nan'),
        'F1 (macro)': float('nan'),
        'MCC': float('nan'),
        'Val Loss (CE)': val_loss
    })

df_summary = pd.DataFrame(summary_rows).round(4)
print(df_summary.to_string(index=False))
df_summary.to_csv(f'{PATH_IMG}/results_summary.csv', index=False)

: 

In [ ]:
# ── Feature importance comparison: TabNet vs TFT ─────────────────────
# Shows top-10 features for each model side by side (binary task)

top10_tabnet = fi_bin.head(10).set_index('feature')['importance']
top10_tft    = fi_tft_bin.head(10).set_index('feature')['importance']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top10_tabnet.plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('TabNet — Top 10 Features (Binary)', fontsize=12)
axes[0].invert_yaxis()

top10_tft.plot.barh(ax=axes[1], color='darkorange')
axes[1].set_title('TFT — Top 10 Features (Binary)', fontsize=12)
axes[1].invert_yaxis()

plt.suptitle('Feature Importance Comparison — Binary Classification', fontsize=14)
plt.tight_layout()
plt.savefig(f'{PATH_IMG}/fi_comparison_binary.png', dpi=150)
plt.show()

: 

In [ ]:
# ── Save best hyperparameters to JSON (reusable) ─────────────────────
all_best_params = {
    'tabnet_binary'    : best_bin,
    'tabnet_multiclass': best_mul,
    'tft_binary'       : best_tft_bin,
    'tft_multiclass'   : best_tft_mul,
}

with open('models/best_hyperparams.json', 'w') as f:
    json.dump(all_best_params, f, indent=2)

print("✅  Best hyperparameters saved to models/best_hyperparams.json")
print(json.dumps(all_best_params, indent=2))

: 

---
## Notes for thesis

### TabNet feature importance
- `feature_importances_` = mean attention mask averaged over all training samples and sequential steps
- Directly interpretable: higher = model relied more on that feature
- Use this to compare importance *before vs after* data perturbation (noise injection / label corruption)

### TFT Variable Selection Network
- `interpret_output()` returns `encoder_variables` and `decoder_variables` weights
- Captures **which features** matter AND **at which time steps**
- Attention weights show temporal patterns: useful for detecting *when* in a flow sequence anomalies emerge

### Recommended noise fragility experiment
1. Train baseline on clean data → record feature importances
2. Inject noise at varying levels (5%, 10%, 20%, 40%) into 🔴 **High fragility** features first
3. Retrain (or fine-tune) and compare feature importance maps
4. Repeat on 🟢 **Low fragility** features as control group
5. Plot importance delta heatmap across noise levels